In [2]:
import sys, torch
print("进程ID:", sys.executable.split('\\')[-2])  # 应该不是 33236
print("PyTorch:", torch.__version__)
from transformers import AutoModelForCausalLM
print("✓ 成功！")

进程ID: nlp-cxy
PyTorch: 2.0.0+cu118
✓ 成功！


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "uer/gpt2-chinese-cluecorpussmall"
print(f"正在加载模型: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("✓ 模型加载成功！")

正在加载模型: uer/gpt2-chinese-cluecorpussmall


c:\Users\15766\anaconda3\envs\nlp-cxy\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/217 [00:00<?, ?B/s]

c:\Users\15766\anaconda3\envs\nlp-cxy\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\15766\.cache\huggingface\hub\models--uer--gpt2-chinese-cluecorpussmall. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/421M [00:00<?, ?B/s]

✓ 模型加载成功！


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "uer/gpt2-chinese-cluecorpussmall"
print(f"正在加载模型: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("✓ 模型加载成功！")

正在加载模型: uer/gpt2-chinese-cluecorpussmall
✓ 模型加载成功！


In [5]:
"""
基于GPT-2的中文文本续写程序
使用Hugging Face的gpt2-chinese-cluecorpussmall模型
根据学号倒数第一位索引选取对应句子开头进行续写
"""

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os

# 设置同步CUDA错误报告
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# 定义10个续写开头（索引0-9）
PROMPTS = {
    0: "如果我拥有一台时间机器",
    1: "当人类第一次踏上火星",
    2: "如果动物会说话，它们最想告诉人类的是",
    3: "有一天，城市突然停电了",
    4: "当我醒来，发现自己变成了一本书",
    5: "假如我能隐身一天，我会",
    6: "我走进了那扇从未打开过的门",
    7: "在一个没有网络的世界里",
    8: "如果世界上只剩下我一个人",
    9: "梦中醒来，一切都变了模样"
}

def load_model(model_name="uer/gpt2-chinese-cluecorpussmall", force_cpu=False):
    """
    加载GPT-2中文模型和分词器
    修复：该模型的自定义CUDA内核与新版PyTorch不兼容，强制使用CPU或修复权重
    """
    print(f"\n正在加载模型: {model_name}")
    
    # 检测CUDA是否可用
    cuda_available = torch.cuda.is_available() and not force_cpu
    device = torch.device("cuda" if cuda_available else "cpu")
    print(f"使用设备: {device}")

    # 加载tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 关键修复：uer/gpt2-chinese-cluecorpussmall的权重文件有问题
    # 必须设置revision和使用特定的加载方式
    print("加载模型权重...")
    
    if cuda_available:
        try:
            # 方案1：尝试直接加载到CUDA（使用float32，不用half）
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                trust_remote_code=True,
                torch_dtype=torch.float32,
                low_cpu_mem_usage=False,  # 关键：禁用低内存模式，避免碎片化
            )
            
            # 检查并修复embedding大小
            vocab_size = len(tokenizer)
            embedding_size = model.get_input_embeddings().num_embeddings
            if vocab_size != embedding_size:
                print(f"调整embedding: {embedding_size} -> {vocab_size}")
                model.resize_token_embeddings(vocab_size)
            
            # 关键修复：在转移前清理CUDA缓存
            torch.cuda.empty_cache()
            torch.cuda.synchronize()  # 同步所有CUDA操作
            
            # 逐层转移而不是整个模型一起转移
            print("转移模型到CUDA...")
            model = model.to(device)
            
        except RuntimeError as e:
            if "CUDA" in str(e):
                print(f"\n⚠️ CUDA错误: {e}")
                print("该模型的自定义CUDA内核与您的PyTorch版本不兼容")
                print("自动切换到CPU模式...\n")
                return load_model(model_name, force_cpu=True)
            raise
    else:
        # CPU模式：稳定但较慢
        print("使用CPU加载（可能需要几秒钟）...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            torch_dtype=torch.float32,
        )
        
        # 同样修复embedding
        if len(tokenizer) != model.get_input_embeddings().num_embeddings:
            model.resize_token_embeddings(len(tokenizer))
        
        model = model.to(device)

    model.eval()
    print(f"✓ 模型加载完成")
    return tokenizer, model, device

def generate_text(prompt, tokenizer, model, device,
                  max_length=200,
                  temperature=0.8,
                  top_k=50,
                  top_p=0.95):
    """生成续写文本"""
    inputs = tokenizer(
        prompt, 
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            no_repeat_ngram_size=2,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

def main():
    student_id = input("请输入你的学号: ").strip()

    if not student_id.isdigit():
        print("错误：学号必须是数字！")
        return

    last_digit = int(student_id[-1])
    prompt = PROMPTS[last_digit]

    print(f"\n{'='*60}")
    print(f"学号: {student_id} | 倒数第一位: {last_digit}")
    print(f"选中的开头: 「{prompt}」")
    print(f"{'='*60}")

    try:
        tokenizer, model, device = load_model()

        print("\n正在生成续写内容...\n")
        result = generate_text(prompt, tokenizer, model, device)

        print(f"{'='*60}")
        print("【续写结果】")
        print(f"{'='*60}")
        print(result)

    except Exception as e:
        print(f"\n✗ 错误: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

错误：学号必须是数字！
